In [1]:
# Cell 1: Mount Drive and install dependencies
from google.colab import drive
drive.mount('/content/drive')

# Install required packages (should mostly be pre-installed)
!pip install torch torchvision torchaudio
!pip install matplotlib numpy pillow scikit-image opencv-python

Mounted at /content/drive


In [2]:
# Cell 2: Create directories
import os
from pathlib import Path

# Create project structure
project_dir = Path('/content/drive/MyDrive/ResearchProject')
project_dir.mkdir(parents=True, exist_ok=True)

# Create subdirectories
(project_dir / 'checkpoints').mkdir(exist_ok=True)
(project_dir / 'logs').mkdir(exist_ok=True)

print("Project structure ready!")
print(f"Project dir: {project_dir}")

Project structure ready!
Project dir: /content/drive/MyDrive/ResearchProject


In [3]:
# Cell 3: Import and setup
import sys
sys.path.append(str('/content/drive/MyDrive/ResearchProject'))

import torch
from unet_denoiser import BlindVideoDenoiserUNet
from dataloader import BlindDenoiseDataset
from unet_denoiser_training import train, TemporalDenoiseDataset, create_train_val_split

# Set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU'}")

# Check GPU memory (important!)
if device == 'cuda':
    print(f"GPU Memory Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Using device: cuda
GPU: NVIDIA A100-SXM4-40GB
GPU Memory Available: 42.4 GB


In [4]:
!cp "/content/drive/MyDrive/ResearchProject/DAVISDataset.zip" /content/
!unzip -q /content/DAVISDataset.zip -d /content/

In [ ]:
from unet_denoiser_training import create_data_loaders, create_train_val_split
train_dataset, val_dataset = create_train_val_split(
    '/content/DAVISDataset', val_split=0.2, seed=42,
    psnr_range=(5, 45), resize_to=(256, 256), use_fp16=True,
    num_input_frames=5, augment=True
)
train_loader, val_loader = create_data_loaders(
    train_dataset, val_dataset, batch_size=16, num_workers=2
)

print(f"\nTrain loader batches per epoch: {len(train_loader)}")
print(f"Val loader batches per epoch: {len(val_loader)}")

Total videos: 150
Train videos: 120 (8714 frames)
Val videos: 30 (2017 frames)
Resolution: 256x256, FP16: True
Noise sampling: PSNR ∈ [5, 45] dB → σ ∈ [1.4, 143.4]
Noise sampling: PSNR ∈ [5, 45] dB → σ ∈ [1.4, 143.4]

DataLoader config:
  Batch size: 16
  Num workers: 2
  Pin memory: True
  Train batches/epoch: 544
  Val batches/epoch: 127

Train loader batches per epoch: 544
Val loader batches per epoch: 127


In [ ]:
model = BlindVideoDenoiserUNet(
    num_input_frames=5,
    base_channels=32
)

print(f"Total model parameters: {sum(p.numel() for p in model.parameters()):,}")

trained_model, logger = train(
    model, train_loader, val_loader,
    num_epochs=100, initial_lr=1e-3, device='cuda',
    checkpoint_dir='/content/drive/MyDrive/ResearchProject/checkpoints_v2',
    log_dir='/content/drive/MyDrive/ResearchProject/logs_v2',
    loss_type='psnr',
    use_amp=True, use_torch_compile=True
)

Total model parameters: 4,674,240


/content/drive/MyDrive/ResearchProject/unet_denoiser_training.py:386: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if (use_amp and device == "cuda") else None


torch.compile enabled (reduce-overhead mode)

Training config: AMP=ON, Loss=psnr, LR=0.001
Epochs: 1-100, Patience: 15



/content/drive/MyDrive/ResearchProject/unet_denoiser_training.py:271: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/content/drive/MyDrive/ResearchProject/unet_denoiser_training.py:271: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/content/drive/MyDrive/ResearchProject/unet_denoiser_training.py:271: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/content/drive/MyDrive/ResearchProject/unet_denoiser_training.py:311: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/content/drive/MyDrive/ResearchProject/unet_denoiser_training.py:311: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch

Epoch    1 | Train Loss: 25.667059 | Val Loss: 24.780550 | LR: 1.00e-03 | Time: 1309.1s
  → Best model saved! (Val Loss: 24.780550)


/content/drive/MyDrive/ResearchProject/unet_denoiser_training.py:271: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/content/drive/MyDrive/ResearchProject/unet_denoiser_training.py:311: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch    2 | Train Loss: 23.361238 | Val Loss: 22.770445 | LR: 1.00e-03 | Time: 1271.9s
  → Best model saved! (Val Loss: 22.770445)


/content/drive/MyDrive/ResearchProject/unet_denoiser_training.py:271: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/content/drive/MyDrive/ResearchProject/unet_denoiser_training.py:311: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch    3 | Train Loss: 22.633943 | Val Loss: 22.281457 | LR: 9.99e-04 | Time: 1263.8s
  → Best model saved! (Val Loss: 22.281457)
Epoch    4 | Train Loss: 22.317909 | Val Loss: 22.063673 | LR: 9.98e-04 | Time: 1262.6s
  → Best model saved! (Val Loss: 22.063673)
Epoch    5 | Train Loss: 22.205442 | Val Loss: 21.704582 | LR: 9.96e-04 | Time: 1277.1s
  → Best model saved! (Val Loss: 21.704582)
Epoch    6 | Train Loss: 22.129118 | Val Loss: 21.835345 | LR: 9.94e-04 | Time: 1271.7s
Epoch    7 | Train Loss: 21.896823 | Val Loss: 22.129538 | LR: 9.91e-04 | Time: 1256.1s
Epoch    8 | Train Loss: 21.704551 | Val Loss: 21.339010 | LR: 9.88e-04 | Time: 1268.1s
  → Best model saved! (Val Loss: 21.339010)
Epoch    9 | Train Loss: 21.710793 | Val Loss: 21.387672 | LR: 9.84e-04 | Time: 1258.7s
Epoch   10 | Train Loss: 21.589177 | Val Loss: 20.956457 | LR: 9.80e-04 | Time: 1259.1s
  Checkpoint saved: /content/drive/MyDrive/ResearchProject/checkpoints_v2/checkpoint_epoch_010.pt
  → Best model saved! 

In [6]:
model = BlindVideoDenoiserUNet(
    num_input_frames=5,
    base_channels=32
)

# Load latest checkpoint
import glob
checkpoint_dir = '/content/drive/MyDrive/ResearchProject/checkpoints_v2'
checkpoints = sorted(glob.glob(f'{checkpoint_dir}/checkpoint_epoch_*.pt'))
latest = checkpoints[-1] if checkpoints else f'{checkpoint_dir}/best_model.pt'

checkpoint = torch.load(latest, map_location='cuda')
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Resuming from epoch {checkpoint['epoch']}, val loss: {checkpoint['val_loss']:.6f}")

print(f"Total model parameters: {sum(p.numel() for p in model.parameters()):,}")

trained_model, logger = train(
    model, train_loader, val_loader,
    num_epochs=100, initial_lr=1e-3, device='cuda',
    checkpoint_dir='/content/drive/MyDrive/ResearchProject/checkpoints_v2',
    log_dir='/content/drive/MyDrive/ResearchProject/logs_v2',
    loss_type='psnr',
    use_amp=True, use_torch_compile=True
)

Resuming from epoch 60, val loss: 19.571072
Total model parameters: 4,674,240


/content/drive/MyDrive/ResearchProject/unet_denoiser_training.py:386: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if (use_amp and device == "cuda") else None


torch.compile enabled (reduce-overhead mode)

Training config: AMP=ON, Loss=psnr, LR=0.001
Epochs: 1-100, Patience: 15



/content/drive/MyDrive/ResearchProject/unet_denoiser_training.py:271: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/content/drive/MyDrive/ResearchProject/unet_denoiser_training.py:271: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/content/drive/MyDrive/ResearchProject/unet_denoiser_training.py:271: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/content/drive/MyDrive/ResearchProject/unet_denoiser_training.py:311: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/content/drive/MyDrive/ResearchProject/unet_denoiser_training.py:311: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch

Epoch    1 | Train Loss: 20.461017 | Val Loss: 20.027272 | LR: 1.00e-03 | Time: 1261.0s
  → Best model saved! (Val Loss: 20.027272)


/content/drive/MyDrive/ResearchProject/unet_denoiser_training.py:271: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/content/drive/MyDrive/ResearchProject/unet_denoiser_training.py:311: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch    2 | Train Loss: 20.447642 | Val Loss: 19.896201 | LR: 1.00e-03 | Time: 1207.8s
  → Best model saved! (Val Loss: 19.896201)


/content/drive/MyDrive/ResearchProject/unet_denoiser_training.py:271: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/content/drive/MyDrive/ResearchProject/unet_denoiser_training.py:311: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch    3 | Train Loss: 20.311364 | Val Loss: 19.862309 | LR: 9.99e-04 | Time: 1210.6s
  → Best model saved! (Val Loss: 19.862309)
Epoch    4 | Train Loss: 20.454418 | Val Loss: 19.936512 | LR: 9.98e-04 | Time: 1205.0s
Epoch    5 | Train Loss: 20.405542 | Val Loss: 20.023486 | LR: 9.96e-04 | Time: 1216.6s
Epoch    6 | Train Loss: 20.310582 | Val Loss: 20.828967 | LR: 9.94e-04 | Time: 1214.7s
Epoch    7 | Train Loss: 20.253089 | Val Loss: 20.088458 | LR: 9.91e-04 | Time: 1204.7s
Epoch    8 | Train Loss: 20.424999 | Val Loss: 20.075685 | LR: 9.88e-04 | Time: 1183.6s
Epoch    9 | Train Loss: 20.369192 | Val Loss: 20.182396 | LR: 9.84e-04 | Time: 1224.0s
Epoch   10 | Train Loss: 20.252059 | Val Loss: 19.869865 | LR: 9.80e-04 | Time: 1205.8s
  Checkpoint saved: /content/drive/MyDrive/ResearchProject/checkpoints_v2/checkpoint_epoch_010.pt
Epoch   11 | Train Loss: 20.307701 | Val Loss: 19.860091 | LR: 9.76e-04 | Time: 1207.8s
  → Best model saved! (Val Loss: 19.860091)
Epoch   12 | Train Los

KeyboardInterrupt: 

In [7]:
# Cell 6: Plot training curves (run this periodically or after training)
from unet_denoiser_training import TrainingLogger
import matplotlib.pyplot as plt

logger = TrainingLogger(log_dir=str(project_dir / 'logs_v2'))
logger.plot_metrics()  # This will save and display the curves

Metrics saved to /content/drive/MyDrive/ResearchProject/logs_v2


In [ ]:
#!cp /content/checkpoints_v2/best_model.pt "/content/drive/MyDrive/ResearchProject/checkpoints_v2/"

In [5]:
# Cell 7: Load best model for inference
model = BlindVideoDenoiserUNet(
    num_input_frames=5,
    base_channels=32
)
checkpoint = torch.load(str(project_dir / 'checkpoints_v2' / 'best_model.pt'), map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Best model loaded from epoch {checkpoint['epoch']}")
print(f"Best val loss: {checkpoint['val_loss']:.6f}")

Best model loaded from epoch 44
Best val loss: 19.430138


In [6]:
# Download FastDVDNet
!git clone https://github.com/m-tassano/fastdvdnet /content/fastdvdnet

# Run benchmark
from benchmark import run_benchmark

results = run_benchmark(
    your_model=model,
    davis_root='/content/DAVISDataset',
    fastdvdnet_repo_path='/content/fastdvdnet',
    save_dir='/content/drive/MyDrive/ResearchProject/denoiser_evaluation/architecture2',
    resize_to=(480, 864)
)

Cloning into '/content/fastdvdnet'...
remote: Enumerating objects: 145, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 145 (delta 21), reused 12 (delta 9), pack-reused 109 (from 1)
Receiving objects: 100% (145/145), 34.97 MiB | 46.75 MiB/s, done.
Resolving deltas: 100% (71/71), done.
Registered denoiser: YourUNet
FastDVDNet loaded from /content/fastdvdnet/model.pth
Registered denoiser: FastDVDNet
Loaded 5 test videos, 50 total frames (864x480)

BENCHMARK: Comparing 2 denoisers across 15 noise levels


--- Noisy PSNR ≈ 5 dB (σ ≈ 143.4) ---
  Noisy baseline:  PSNR=8.65 dB, SSIM=0.0423
  YourUNet            : PSNR=23.83 dB (+15.18), SSIM=0.6410, Time=2.5s
  FastDVDNet          : PSNR=18.13 dB (+9.48), SSIM=0.3495, Time=1.7s
  VRT (published)     : N/A
  FastDVDNet (pub.)   : N/A

--- Noisy PSNR ≈ 8 dB (σ ≈ 101.5) ---
  Noisy baseline:  PSNR=10.31 dB, SSIM=0.0655
  YourUNet            : PSNR=25.73 dB (+15.42), SSIM=0.7106,